In [1]:
#Find the top 3 products by revenue in each region (RANK vs DENSE_RANK vs ROW_NUMBER — explain the difference).

In [3]:
from pyspark.sql import SparkSession

from pyspark.sql.functions import col, sum as spark_sum
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, rank, dense_rank
spark=SparkSession.builder.appName("window").getOrCreate()
sales_data = [
    ("East", "Laptop", 1000),
    ("East", "Laptop", 1500),
    ("East", "Mobile", 2000),
    ("East", "Tablet", 2000),
    ("East", "Watch", 800),

    ("West", "Laptop", 3000),
    ("West", "Mobile", 3000),
    ("West", "Tablet", 1500),
    ("West", "Watch", 1000),
    ("West", "Camera", 500),

    ("North", "Laptop", 2500),
    ("North", "Mobile", 1800),
    ("North", "Tablet", 1800),
    ("North", "Watch", 900)
]

columns = ["region", "product", "revenue"]

sales_df = spark.createDataFrame(sales_data, columns)

sales_df.show()

26/06/13 05:23:48 WARN Utils: Your hostname, MKHOMBP14018.local resolves to a loopback address: 127.0.0.1; using 192.168.1.38 instead (on interface en0)
26/06/13 05:23:48 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/13 05:23:52 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/13 05:23:52 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
                                                                                

+------+-------+-------+
|region|product|revenue|
+------+-------+-------+
|  East| Laptop|   1000|
|  East| Laptop|   1500|
|  East| Mobile|   2000|
|  East| Tablet|   2000|
|  East|  Watch|    800|
|  West| Laptop|   3000|
|  West| Mobile|   3000|
|  West| Tablet|   1500|
|  West|  Watch|   1000|
|  West| Camera|    500|
| North| Laptop|   2500|
| North| Mobile|   1800|
| North| Tablet|   1800|
| North|  Watch|    900|
+------+-------+-------+



In [4]:
product_revenue_df = (
    sales_df
    .groupBy("region", "product")
    .agg(spark_sum("revenue").alias("total_revenue"))
)

product_revenue_df.show()

+------+-------+-------------+
|region|product|total_revenue|
+------+-------+-------------+
|  East| Laptop|         2500|
|  East| Mobile|         2000|
|  East|  Watch|          800|
|  East| Tablet|         2000|
|  West| Mobile|         3000|
|  West| Laptop|         3000|
|  West| Tablet|         1500|
|  West|  Watch|         1000|
|  West| Camera|          500|
| North| Laptop|         2500|
| North| Mobile|         1800|
| North| Tablet|         1800|
| North|  Watch|          900|
+------+-------+-------------+



In [5]:
window_spec = Window.partitionBy("region").orderBy(col("total_revenue").desc())

ranked_df = (
    product_revenue_df
    .withColumn("row_number", row_number().over(window_spec))
    .withColumn("rank", rank().over(window_spec))
    .withColumn("dense_rank", dense_rank().over(window_spec))
)

ranked_df.orderBy("region", "row_number").show()

+------+-------+-------------+----------+----+----------+
|region|product|total_revenue|row_number|rank|dense_rank|
+------+-------+-------------+----------+----+----------+
|  East| Laptop|         2500|         1|   1|         1|
|  East| Mobile|         2000|         2|   2|         2|
|  East| Tablet|         2000|         3|   2|         2|
|  East|  Watch|          800|         4|   4|         3|
| North| Laptop|         2500|         1|   1|         1|
| North| Mobile|         1800|         2|   2|         2|
| North| Tablet|         1800|         3|   2|         2|
| North|  Watch|          900|         4|   4|         3|
|  West| Mobile|         3000|         1|   1|         1|
|  West| Laptop|         3000|         2|   1|         1|
|  West| Tablet|         1500|         3|   3|         2|
|  West|  Watch|         1000|         4|   4|         3|
|  West| Camera|          500|         5|   5|         4|
+------+-------+-------------+----------+----+----------+



In [8]:
sales_df.createOrReplaceTempView("sales")

In [18]:
sql1="""
WITH product_revenue AS (
    SELECT
        region,
        product,
        SUM(revenue) AS total_revenue
    FROM sales
    GROUP BY region, product
),
ranked AS (
    SELECT
        region,
        product,
        total_revenue,
        ROW_NUMBER() OVER (
            PARTITION BY region
            ORDER BY total_revenue DESC
        ) AS row_num1,

            RANK() OVER (
            PARTITION BY region
            ORDER BY total_revenue DESC
        ) AS rank_num1,

             DENSE_RANK() OVER (
            PARTITION BY region
            ORDER BY total_revenue DESC
        ) AS dense_num1
        
    FROM product_revenue
)
SELECT
    region,
    product,
    total_revenue,
    row_num1,
    rank_num1,
    dense_num1
FROM ranked
"""
spark.sql(sql1).show()

+------+-------+-------------+--------+---------+----------+
|region|product|total_revenue|row_num1|rank_num1|dense_num1|
+------+-------+-------------+--------+---------+----------+
|  East| Laptop|         2500|       1|        1|         1|
|  East| Mobile|         2000|       2|        2|         2|
|  East| Tablet|         2000|       3|        2|         2|
|  East|  Watch|          800|       4|        4|         3|
| North| Laptop|         2500|       1|        1|         1|
| North| Mobile|         1800|       2|        2|         2|
| North| Tablet|         1800|       3|        2|         2|
| North|  Watch|          900|       4|        4|         3|
|  West| Mobile|         3000|       1|        1|         1|
|  West| Laptop|         3000|       2|        1|         1|
|  West| Tablet|         1500|       3|        3|         2|
|  West|  Watch|         1000|       4|        4|         3|
|  West| Camera|          500|       5|        5|         4|
+------+-------+--------

In [19]:
#Find each employee's salary rank within their department.


In [20]:
employees_data = [

    (1, "Alice", "IT", 90000),

    (2, "Bob", "IT", 70000),

    (3, "Charlie", "IT", 90000),

    (4, "David", "HR", 60000),

    (5, "Eva", "HR", 80000),

    (6, "Frank", "HR", 80000),

    (7, "Grace", "Finance", 95000),

    (8, "Henry", "Finance", 75000),

    (9, "Ivy", "Finance", 65000)

]

columns = ["emp_id", "emp_name", "department", "salary"]

employees_df = spark.createDataFrame(employees_data, columns)

employees_df.show()

+------+--------+----------+------+
|emp_id|emp_name|department|salary|
+------+--------+----------+------+
|     1|   Alice|        IT| 90000|
|     2|     Bob|        IT| 70000|
|     3| Charlie|        IT| 90000|
|     4|   David|        HR| 60000|
|     5|     Eva|        HR| 80000|
|     6|   Frank|        HR| 80000|
|     7|   Grace|   Finance| 95000|
|     8|   Henry|   Finance| 75000|
|     9|     Ivy|   Finance| 65000|
+------+--------+----------+------+



In [26]:
employees_df.createOrReplaceTempView("employees")


In [29]:
sql2 ="""

select emp_id, emp_name, department, salary, rank() over (partition by department order by salary DESC ) rank from employees

"""
spark.sql(sql2).show()

+------+--------+----------+------+----+
|emp_id|emp_name|department|salary|rank|
+------+--------+----------+------+----+
|     7|   Grace|   Finance| 95000|   1|
|     8|   Henry|   Finance| 75000|   2|
|     9|     Ivy|   Finance| 65000|   3|
|     5|     Eva|        HR| 80000|   1|
|     6|   Frank|        HR| 80000|   1|
|     4|   David|        HR| 60000|   3|
|     1|   Alice|        IT| 90000|   1|
|     3| Charlie|        IT| 90000|   1|
|     2|     Bob|        IT| 70000|   3|
+------+--------+----------+------+----+



In [30]:
from pyspark.sql.functions import col
from pyspark.sql.window import Window
from pyspark.sql.functions import rank, dense_rank, row_number

employees_data = [
    (1, "Alice", "IT", 90000),
    (2, "Bob", "IT", 70000),
    (3, "Charlie", "IT", 90000),
    (4, "David", "HR", 60000),
    (5, "Eva", "HR", 80000),
    (6, "Frank", "HR", 80000),
    (7, "Grace", "Finance", 95000),
    (8, "Henry", "Finance", 75000),
    (9, "Ivy", "Finance", 65000)
]

columns = ["emp_id", "emp_name", "department", "salary"]

employees_df = spark.createDataFrame(employees_data, columns)

window_spec = Window.partitionBy("department").orderBy(col("salary").desc())

result = (
    employees_df
    .withColumn("salary_rank", rank().over(window_spec))
    .orderBy("department", "salary_rank")
)

result.show()

+------+--------+----------+------+-----------+
|emp_id|emp_name|department|salary|salary_rank|
+------+--------+----------+------+-----------+
|     7|   Grace|   Finance| 95000|          1|
|     8|   Henry|   Finance| 75000|          2|
|     9|     Ivy|   Finance| 65000|          3|
|     5|     Eva|        HR| 80000|          1|
|     6|   Frank|        HR| 80000|          1|
|     4|   David|        HR| 60000|          3|
|     1|   Alice|        IT| 90000|          1|
|     3| Charlie|        IT| 90000|          1|
|     2|     Bob|        IT| 70000|          3|
+------+--------+----------+------+-----------+



In [31]:
#Compute a 7-day moving average of daily sales.


In [32]:
from pyspark.sql.functions import col, to_date, avg
from pyspark.sql.window import Window

sales_data = [
    ("2026-01-01", 100),
    ("2026-01-02", 120),
    ("2026-01-03", 130),
    ("2026-01-04", 110),
    ("2026-01-05", 150),
    ("2026-01-06", 170),
    ("2026-01-07", 160),
    ("2026-01-08", 180),
    ("2026-01-09", 200),
    ("2026-01-10", 190)
]

columns = ["sale_date", "daily_sales"]

sales_df = spark.createDataFrame(sales_data, columns)

sales_df = sales_df.withColumn("sale_date", to_date(col("sale_date")))

sales_df.show()

+----------+-----------+
| sale_date|daily_sales|
+----------+-----------+
|2026-01-01|        100|
|2026-01-02|        120|
|2026-01-03|        130|
|2026-01-04|        110|
|2026-01-05|        150|
|2026-01-06|        170|
|2026-01-07|        160|
|2026-01-08|        180|
|2026-01-09|        200|
|2026-01-10|        190|
+----------+-----------+



In [42]:
sales_df.createOrReplaceTempView("daily_sales")

In [43]:
sql3 ="""

SELECT

    sale_date,

    daily_sales,

    ROUND(

        AVG(daily_sales) OVER (

            ORDER BY sale_date

            ROWS BETWEEN 6 PRECEDING AND CURRENT ROW

        ),

        2

    ) AS moving_avg_7_day

FROM daily_sales

ORDER BY sale_date
"""
spark.sql(sql3).show()

+----------+-----------+----------------+
| sale_date|daily_sales|moving_avg_7_day|
+----------+-----------+----------------+
|2026-01-01|        100|           100.0|
|2026-01-02|        120|           110.0|
|2026-01-03|        130|          116.67|
|2026-01-04|        110|           115.0|
|2026-01-05|        150|           122.0|
|2026-01-06|        170|           130.0|
|2026-01-07|        160|          134.29|
|2026-01-08|        180|          145.71|
|2026-01-09|        200|          157.14|
|2026-01-10|        190|          165.71|
+----------+-----------+----------------+



26/06/13 05:47:27 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 05:47:27 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 05:47:27 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 05:47:27 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 05:47:27 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


In [45]:
from pyspark.sql.functions import col, to_date, avg, round
from pyspark.sql.window import Window

sales_data = [
    ("2026-01-01", 100),
    ("2026-01-02", 120),
    ("2026-01-03", 130),
    ("2026-01-04", 110),
    ("2026-01-05", 150),
    ("2026-01-06", 170),
    ("2026-01-07", 160),
    ("2026-01-08", 180),
    ("2026-01-09", 200),
    ("2026-01-10", 190)
]

columns = ["sale_date", "daily_sales"]

sales_df = spark.createDataFrame(sales_data, columns)

sales_df = sales_df.withColumn("sale_date", to_date(col("sale_date")))

window_spec = (
    Window
    .orderBy("sale_date")
    .rowsBetween(-6, 0)
)

result = (
    sales_df
    .withColumn("moving_avg_7_day", round(avg("daily_sales").over(window_spec), 2))
    .orderBy("sale_date")
)

result.show()

+----------+-----------+----------------+
| sale_date|daily_sales|moving_avg_7_day|
+----------+-----------+----------------+
|2026-01-01|        100|           100.0|
|2026-01-02|        120|           110.0|
|2026-01-03|        130|          116.67|
|2026-01-04|        110|           115.0|
|2026-01-05|        150|           122.0|
|2026-01-06|        170|           130.0|
|2026-01-07|        160|          134.29|
|2026-01-08|        180|          145.71|
|2026-01-09|        200|          157.14|
|2026-01-10|        190|          165.71|
+----------+-----------+----------------+



26/06/13 05:49:46 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 05:49:46 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 05:49:46 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 05:49:46 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 05:49:46 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


In [ ]:
#Compute a running (cumulative) total of revenue by month.


In [46]:
from pyspark.sql.functions import col, to_date, date_format, sum as spark_sum
from pyspark.sql.window import Window

sales_data = [
    ("2026-01-05", 1000),
    ("2026-01-12", 1500),
    ("2026-02-03", 2000),
    ("2026-02-18", 2500),
    ("2026-03-07", 3000),
    ("2026-03-21", 3500),
    ("2026-04-10", 4000)
]

columns = ["sale_date", "revenue"]

sales_df = spark.createDataFrame(sales_data, columns)

sales_df = sales_df.withColumn("sale_date", to_date(col("sale_date")))

sales_df.show()

+----------+-------+
| sale_date|revenue|
+----------+-------+
|2026-01-05|   1000|
|2026-01-12|   1500|
|2026-02-03|   2000|
|2026-02-18|   2500|
|2026-03-07|   3000|
|2026-03-21|   3500|
|2026-04-10|   4000|
+----------+-------+



In [47]:
monthly_revenue_df = (
    sales_df
    .withColumn("month", date_format(col("sale_date"), "yyyy-MM"))
    .groupBy("month")
    .agg(spark_sum("revenue").alias("monthly_revenue"))
)

monthly_revenue_df.show()

+-------+---------------+
|  month|monthly_revenue|
+-------+---------------+
|2026-01|           2500|
|2026-02|           4500|
|2026-03|           6500|
|2026-04|           4000|
+-------+---------------+



In [48]:
window_spec = (
    Window
    .orderBy("month")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

result = (
    monthly_revenue_df
    .withColumn("running_total_revenue", spark_sum("monthly_revenue").over(window_spec))
    .orderBy("month")
)

result.show()

+-------+---------------+---------------------+
|  month|monthly_revenue|running_total_revenue|
+-------+---------------+---------------------+
|2026-01|           2500|                 2500|
|2026-02|           4500|                 7000|
|2026-03|           6500|                13500|
|2026-04|           4000|                17500|
+-------+---------------+---------------------+



26/06/13 05:58:17 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 05:58:17 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 05:58:17 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 05:58:17 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 05:58:17 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 05:58:17 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 0

In [49]:
sales_df.createOrReplaceTempView("sales")

In [51]:
sql3="""
WITH monthly_revenue AS (
    SELECT
        DATE_FORMAT(sale_date, 'yyyy-MM') AS month,
        SUM(revenue) AS monthly_revenue
    FROM sales
    GROUP BY DATE_FORMAT(sale_date, 'yyyy-MM')
)

SELECT
    month,
    monthly_revenue,
    SUM(monthly_revenue) OVER (
        ORDER BY month
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS running_total_revenue
FROM monthly_revenue
ORDER BY month;

"""
spark.sql(sql3).show()

26/06/13 06:00:14 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 06:00:14 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 06:00:14 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 06:00:14 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 06:00:14 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 06:00:14 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 0

+-------+---------------+---------------------+
|  month|monthly_revenue|running_total_revenue|
+-------+---------------+---------------------+
|2026-01|           2500|                 2500|
|2026-02|           4500|                 7000|
|2026-03|           6500|                13500|
|2026-04|           4000|                17500|
+-------+---------------+---------------------+



In [53]:
#For each user, find the time difference between consecutive logins (LAG/LEAD).


In [52]:
from pyspark.sql.functions import col, to_timestamp
from pyspark.sql.window import Window
from pyspark.sql.functions import lag, lead, unix_timestamp, round

login_data = [
    (1, "Alice", "2026-01-01 09:00:00"),
    (1, "Alice", "2026-01-01 12:30:00"),
    (1, "Alice", "2026-01-02 10:00:00"),
    (1, "Alice", "2026-01-05 08:00:00"),

    (2, "Bob", "2026-01-01 10:15:00"),
    (2, "Bob", "2026-01-03 11:45:00"),
    (2, "Bob", "2026-01-03 15:00:00"),

    (3, "Charlie", "2026-01-02 14:00:00")
]

columns = ["user_id", "user_name", "login_time"]

login_df = spark.createDataFrame(login_data, columns)

login_df = login_df.withColumn(
    "login_time",
    to_timestamp(col("login_time"))
)

login_df.show(truncate=False)

+-------+---------+-------------------+
|user_id|user_name|login_time         |
+-------+---------+-------------------+
|1      |Alice    |2026-01-01 09:00:00|
|1      |Alice    |2026-01-01 12:30:00|
|1      |Alice    |2026-01-02 10:00:00|
|1      |Alice    |2026-01-05 08:00:00|
|2      |Bob      |2026-01-01 10:15:00|
|2      |Bob      |2026-01-03 11:45:00|
|2      |Bob      |2026-01-03 15:00:00|
|3      |Charlie  |2026-01-02 14:00:00|
+-------+---------+-------------------+



In [71]:
login_df.createOrReplaceTempView("user_logins")

In [75]:
sql4="""
WITH login_with_prev_next AS (
    SELECT
        user_id,
        user_name,
        login_time,

        LAG(login_time) OVER (
            PARTITION BY user_id
            ORDER BY login_time
        ) AS previous_login_time,

        LEAD(login_time) OVER (
            PARTITION BY user_id
            ORDER BY login_time
        ) AS next_login_time

    FROM user_logins
)

SELECT
    user_id,
    user_name,
    login_time,
    previous_login_time,
    next_login_time

FROM login_with_prev_next
ORDER BY user_id, login_time;

"""
spark.sql(sql4).show()

+-------+---------+-------------------+-------------------+-------------------+
|user_id|user_name|         login_time|previous_login_time|    next_login_time|
+-------+---------+-------------------+-------------------+-------------------+
|      1|    Alice|2026-01-01 09:00:00|               NULL|2026-01-01 12:30:00|
|      1|    Alice|2026-01-01 12:30:00|2026-01-01 09:00:00|2026-01-02 10:00:00|
|      1|    Alice|2026-01-02 10:00:00|2026-01-01 12:30:00|2026-01-05 08:00:00|
|      1|    Alice|2026-01-05 08:00:00|2026-01-02 10:00:00|               NULL|
|      2|      Bob|2026-01-01 10:15:00|               NULL|2026-01-03 11:45:00|
|      2|      Bob|2026-01-03 11:45:00|2026-01-01 10:15:00|2026-01-03 15:00:00|
|      2|      Bob|2026-01-03 15:00:00|2026-01-03 11:45:00|               NULL|
|      3|  Charlie|2026-01-02 14:00:00|               NULL|               NULL|
+-------+---------+-------------------+-------------------+-------------------+



In [76]:
sql4="""
SELECT

    user_id,

    user_name,

    login_time,

    LAG(login_time) OVER (

        PARTITION BY user_id

        ORDER BY login_time

    ) AS previous_login_time,

    ROUND(

        (

            UNIX_TIMESTAMP(login_time)

            -

            UNIX_TIMESTAMP(

                LAG(login_time) OVER (

                    PARTITION BY user_id

                    ORDER BY login_time

                )

            )

        ) / 3600,

        2

    ) AS hours_since_previous_login,

    LEAD(login_time) OVER (

        PARTITION BY user_id

        ORDER BY login_time

    ) AS next_login_time,

    ROUND(

        (

            UNIX_TIMESTAMP(

                LEAD(login_time) OVER (

                    PARTITION BY user_id

                    ORDER BY login_time

                )

            )

            -

            UNIX_TIMESTAMP(login_time)

        ) / 3600,

        2

    ) AS hours_until_next_login

FROM user_logins

ORDER BY user_id, login_time

"""
spark.sql(sql4).show()

+-------+---------+-------------------+-------------------+--------------------------+-------------------+----------------------+
|user_id|user_name|         login_time|previous_login_time|hours_since_previous_login|    next_login_time|hours_until_next_login|
+-------+---------+-------------------+-------------------+--------------------------+-------------------+----------------------+
|      1|    Alice|2026-01-01 09:00:00|               NULL|                      NULL|2026-01-01 12:30:00|                   3.5|
|      1|    Alice|2026-01-01 12:30:00|2026-01-01 09:00:00|                       3.5|2026-01-02 10:00:00|                  21.5|
|      1|    Alice|2026-01-02 10:00:00|2026-01-01 12:30:00|                      21.5|2026-01-05 08:00:00|                  70.0|
|      1|    Alice|2026-01-05 08:00:00|2026-01-02 10:00:00|                      70.0|               NULL|                  NULL|
|      2|      Bob|2026-01-01 10:15:00|               NULL|                      NULL|2026

In [77]:
from pyspark.sql.functions import col, to_timestamp
from pyspark.sql.window import Window
from pyspark.sql.functions import lag, lead, unix_timestamp, round

login_data = [
    (1, "Alice", "2026-01-01 09:00:00"),
    (1, "Alice", "2026-01-01 12:30:00"),
    (1, "Alice", "2026-01-02 10:00:00"),
    (1, "Alice", "2026-01-05 08:00:00"),

    (2, "Bob", "2026-01-01 10:15:00"),
    (2, "Bob", "2026-01-03 11:45:00"),
    (2, "Bob", "2026-01-03 15:00:00"),

    (3, "Charlie", "2026-01-02 14:00:00")
]

columns = ["user_id", "user_name", "login_time"]

login_df = spark.createDataFrame(login_data, columns)

login_df = login_df.withColumn(
    "login_time",
    to_timestamp(col("login_time"))
)

window_spec = Window.partitionBy("user_id").orderBy("login_time")

result = (
    login_df
    .withColumn("previous_login_time", lag("login_time").over(window_spec))
    .withColumn("next_login_time", lead("login_time").over(window_spec))
    .withColumn(
        "hours_since_previous_login",
        round(
            (unix_timestamp("login_time") - unix_timestamp("previous_login_time")) / 3600,
            2
        )
    )
    .withColumn(
        "hours_until_next_login",
        round(
            (unix_timestamp("next_login_time") - unix_timestamp("login_time")) / 3600,
            2
        )
    )
    .orderBy("user_id", "login_time")
)

result.show(truncate=False)

+-------+---------+-------------------+-------------------+-------------------+--------------------------+----------------------+
|user_id|user_name|login_time         |previous_login_time|next_login_time    |hours_since_previous_login|hours_until_next_login|
+-------+---------+-------------------+-------------------+-------------------+--------------------------+----------------------+
|1      |Alice    |2026-01-01 09:00:00|NULL               |2026-01-01 12:30:00|NULL                      |3.5                   |
|1      |Alice    |2026-01-01 12:30:00|2026-01-01 09:00:00|2026-01-02 10:00:00|3.5                       |21.5                  |
|1      |Alice    |2026-01-02 10:00:00|2026-01-01 12:30:00|2026-01-05 08:00:00|21.5                      |70.0                  |
|1      |Alice    |2026-01-05 08:00:00|2026-01-02 10:00:00|NULL               |70.0                      |NULL                  |
|2      |Bob      |2026-01-01 10:15:00|NULL               |2026-01-03 11:45:00|NULL       

In [78]:
#Find the second-highest salary — (a) with a window function, (b) without one.

In [79]:
employees_data = [
    (1, "Alice", 90000),
    (2, "Bob", 70000),
    (3, "Charlie", 95000),
    (4, "David", 60000),
    (5, "Eva", 80000),
    (6, "Frank", 95000),
    (7, "Grace", 70000)
]

columns = ["emp_id", "emp_name", "salary"]

employees_df = spark.createDataFrame(employees_data, columns)

employees_df.show()

+------+--------+------+
|emp_id|emp_name|salary|
+------+--------+------+
|     1|   Alice| 90000|
|     2|     Bob| 70000|
|     3| Charlie| 95000|
|     4|   David| 60000|
|     5|     Eva| 80000|
|     6|   Frank| 95000|
|     7|   Grace| 70000|
+------+--------+------+



In [80]:
from pyspark.sql.functions import col, dense_rank
from pyspark.sql.window import Window

window_spec = Window.orderBy(col("salary").desc())

ranked_df = employees_df.withColumn(
    "salary_rank",
    dense_rank().over(window_spec)
)

result = ranked_df.filter(col("salary_rank") == 2)

result.show()

+------+--------+------+-----------+
|emp_id|emp_name|salary|salary_rank|
+------+--------+------+-----------+
|     1|   Alice| 90000|          2|
+------+--------+------+-----------+



26/06/13 06:13:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 06:13:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 06:13:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 06:13:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 06:13:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


In [81]:
employees_df.createGlobalTempView("employees")

In [85]:
sql5="""WITH ranked AS (
    SELECT
        emp_id,
        emp_name,
        salary,
        DENSE_RANK() OVER (
            ORDER BY salary DESC
        ) AS salary_rank
    FROM employees
)

SELECT
    emp_id,
    emp_name,
    salary,
    salary_rank
FROM ranked
WHERE salary_rank = 2
"""
spark.sql(sql5).show(truncate=False)

+------+--------+------+-----------+
|emp_id|emp_name|salary|salary_rank|
+------+--------+------+-----------+
|1     |Alice   |90000 |2          |
|3     |Charlie |90000 |2          |
+------+--------+------+-----------+



26/06/13 06:16:12 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 06:16:12 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 06:16:12 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 06:16:12 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 06:16:12 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


In [86]:
#Compute month-over-month % growth in revenue.


In [87]:
from pyspark.sql.functions import col, to_date, date_format, sum as spark_sum
from pyspark.sql.window import Window
from pyspark.sql.functions import lag, round

sales_data = [
    ("2026-01-05", 1000),
    ("2026-01-12", 9000),
    ("2026-02-03", 12000),
    ("2026-03-07", 15000),
    ("2026-04-10", 13500),
    ("2026-05-18", 18000)
]

columns = ["sale_date", "revenue"]

sales_df = spark.createDataFrame(sales_data, columns)

sales_df = sales_df.withColumn("sale_date", to_date(col("sale_date")))

sales_df.show()

+----------+-------+
| sale_date|revenue|
+----------+-------+
|2026-01-05|   1000|
|2026-01-12|   9000|
|2026-02-03|  12000|
|2026-03-07|  15000|
|2026-04-10|  13500|
|2026-05-18|  18000|
+----------+-------+



In [88]:
monthly_revenue_df = (
    sales_df
    .withColumn("month", date_format(col("sale_date"), "yyyy-MM"))
    .groupBy("month")
    .agg(spark_sum("revenue").alias("monthly_revenue"))
)

monthly_revenue_df.show()

+-------+---------------+
|  month|monthly_revenue|
+-------+---------------+
|2026-01|          10000|
|2026-02|          12000|
|2026-03|          15000|
|2026-04|          13500|
|2026-05|          18000|
+-------+---------------+



In [90]:
window_spec = Window.orderBy("month")

result = (
    monthly_revenue_df
    .withColumn("previous_month_revenue", lag("monthly_revenue").over(window_spec))
    .withColumn(
        "mom_growth_percent",
        round(
            ((col("monthly_revenue") - col("previous_month_revenue")) / col("previous_month_revenue")) * 100,
            2
        )
    )
    .orderBy("month")
)

result.show()

+-------+---------------+----------------------+------------------+
|  month|monthly_revenue|previous_month_revenue|mom_growth_percent|
+-------+---------------+----------------------+------------------+
|2026-01|          10000|                  NULL|              NULL|
|2026-02|          12000|                 10000|              20.0|
|2026-03|          15000|                 12000|              25.0|
|2026-04|          13500|                 15000|             -10.0|
|2026-05|          18000|                 13500|             33.33|
+-------+---------------+----------------------+------------------+



26/06/13 06:17:57 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 06:17:57 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 06:17:57 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 06:17:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 06:17:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 06:17:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 0

In [91]:
sales_df.createOrReplaceTempView("sales")

In [94]:
sql6="""

WITH monthly_revenue AS (
    SELECT
        DATE_FORMAT(sale_date, 'yyyy-MM') AS month,
        SUM(revenue) AS monthly_revenue
    FROM sales
    GROUP BY DATE_FORMAT(sale_date, 'yyyy-MM')
),

with_previous_month AS (
    SELECT
        month,
        monthly_revenue,
        LAG(monthly_revenue) OVER (
            ORDER BY month
        ) AS previous_month_revenue
    FROM monthly_revenue
)

SELECT
    month,
    monthly_revenue,
    previous_month_revenue,
    CASE
        WHEN previous_month_revenue IS NULL THEN NULL
        WHEN previous_month_revenue = 0 THEN NULL
        ELSE ROUND(
            ((monthly_revenue - previous_month_revenue) / previous_month_revenue) * 100,
            2
        )
    END AS mom_growth_percent
FROM with_previous_month
ORDER BY month;
"""
spark.sql(sql6).show()

+-------+---------------+----------------------+------------------+
|  month|monthly_revenue|previous_month_revenue|mom_growth_percent|
+-------+---------------+----------------------+------------------+
|2026-01|          10000|                  NULL|              NULL|
|2026-02|          12000|                 10000|              20.0|
|2026-03|          15000|                 12000|              25.0|
|2026-04|          13500|                 15000|             -10.0|
|2026-05|          18000|                 13500|             33.33|
+-------+---------------+----------------------+------------------+



26/06/13 06:20:23 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 06:20:23 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 06:20:23 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 06:20:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 06:20:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 06:20:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 0

In [95]:
from pyspark.sql.functions import col, to_date, date_format, sum as spark_sum, lag, round, when
from pyspark.sql.window import Window

sales_data = [
    ("2026-01-05", 1000),
    ("2026-01-12", 9000),
    ("2026-02-03", 12000),
    ("2026-03-07", 15000),
    ("2026-04-10", 13500),
    ("2026-05-18", 18000)
]

columns = ["sale_date", "revenue"]

sales_df = spark.createDataFrame(sales_data, columns)

sales_df = sales_df.withColumn("sale_date", to_date(col("sale_date")))

monthly_revenue_df = (
    sales_df
    .withColumn("month", date_format(col("sale_date"), "yyyy-MM"))
    .groupBy("month")
    .agg(spark_sum("revenue").alias("monthly_revenue"))
)

window_spec = Window.orderBy("month")

result = (
    monthly_revenue_df
    .withColumn("previous_month_revenue", lag("monthly_revenue").over(window_spec))
    .withColumn(
        "mom_growth_percent",
        when(col("previous_month_revenue").isNull(), None)
        .when(col("previous_month_revenue") == 0, None)
        .otherwise(
            round(
                ((col("monthly_revenue") - col("previous_month_revenue")) / col("previous_month_revenue")) * 100,
                2
            )
        )
    )
    .orderBy("month")
)

result.show()

+-------+---------------+----------------------+------------------+
|  month|monthly_revenue|previous_month_revenue|mom_growth_percent|
+-------+---------------+----------------------+------------------+
|2026-01|          10000|                  NULL|              NULL|
|2026-02|          12000|                 10000|              20.0|
|2026-03|          15000|                 12000|              25.0|
|2026-04|          13500|                 15000|             -10.0|
|2026-05|          18000|                 13500|             33.33|
+-------+---------------+----------------------+------------------+



26/06/13 06:20:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 06:20:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 06:20:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 06:20:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 06:20:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 06:20:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 0

In [96]:
#For each customer, flag their first and most recent purchase.


In [97]:
from pyspark.sql.functions import col, to_timestamp, when
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

purchase_data = [
    (101, "C001", "Alice", "2026-01-05 10:00:00", 500),
    (102, "C001", "Alice", "2026-01-10 14:30:00", 300),
    (103, "C001", "Alice", "2026-02-01 09:15:00", 700),

    (104, "C002", "Bob", "2026-01-07 11:00:00", 200),
    (105, "C002", "Bob", "2026-01-20 16:45:00", 400),

    (106, "C003", "Charlie", "2026-02-05 13:00:00", 900)
]

columns = ["order_id", "customer_id", "customer_name", "purchase_time", "amount"]

purchases_df = spark.createDataFrame(purchase_data, columns)

purchases_df = purchases_df.withColumn(
    "purchase_time",
    to_timestamp(col("purchase_time"))
)

purchases_df.show(truncate=False)

+--------+-----------+-------------+-------------------+------+
|order_id|customer_id|customer_name|purchase_time      |amount|
+--------+-----------+-------------+-------------------+------+
|101     |C001       |Alice        |2026-01-05 10:00:00|500   |
|102     |C001       |Alice        |2026-01-10 14:30:00|300   |
|103     |C001       |Alice        |2026-02-01 09:15:00|700   |
|104     |C002       |Bob          |2026-01-07 11:00:00|200   |
|105     |C002       |Bob          |2026-01-20 16:45:00|400   |
|106     |C003       |Charlie      |2026-02-05 13:00:00|900   |
+--------+-----------+-------------+-------------------+------+



In [99]:
first_purchase_window = (
    Window
    .partitionBy("customer_id")
    .orderBy(col("purchase_time").asc(), col("order_id").asc())
)

latest_purchase_window = (
    Window
    .partitionBy("customer_id")
    .orderBy(col("purchase_time").desc(), col("order_id").desc())
)

ranked_df = (
    purchases_df
    .withColumn("first_purchase_rn", row_number().over(first_purchase_window))
    .withColumn("latest_purchase_rn", row_number().over(latest_purchase_window))
)

ranked_df.show(truncate=False)

+--------+-----------+-------------+-------------------+------+-----------------+------------------+
|order_id|customer_id|customer_name|purchase_time      |amount|first_purchase_rn|latest_purchase_rn|
+--------+-----------+-------------+-------------------+------+-----------------+------------------+
|103     |C001       |Alice        |2026-02-01 09:15:00|700   |3                |1                 |
|102     |C001       |Alice        |2026-01-10 14:30:00|300   |2                |2                 |
|101     |C001       |Alice        |2026-01-05 10:00:00|500   |1                |3                 |
|105     |C002       |Bob          |2026-01-20 16:45:00|400   |2                |1                 |
|104     |C002       |Bob          |2026-01-07 11:00:00|200   |1                |2                 |
|106     |C003       |Charlie      |2026-02-05 13:00:00|900   |1                |1                 |
+--------+-----------+-------------+-------------------+------+-----------------+----------

In [100]:
result = (
    ranked_df
    .withColumn(
        "is_first_purchase",
        when(col("first_purchase_rn") == 1, 1).otherwise(0)
    )
    .withColumn(
        "is_most_recent_purchase",
        when(col("latest_purchase_rn") == 1, 1).otherwise(0)
    )
    .select(
        "order_id",
        "customer_id",
        "customer_name",
        "purchase_time",
        "amount",
        "is_first_purchase",
        "is_most_recent_purchase"
    )
    .orderBy("customer_id", "purchase_time")
)

result.show(truncate=False)

+--------+-----------+-------------+-------------------+------+-----------------+-----------------------+
|order_id|customer_id|customer_name|purchase_time      |amount|is_first_purchase|is_most_recent_purchase|
+--------+-----------+-------------+-------------------+------+-----------------+-----------------------+
|101     |C001       |Alice        |2026-01-05 10:00:00|500   |1                |0                      |
|102     |C001       |Alice        |2026-01-10 14:30:00|300   |0                |0                      |
|103     |C001       |Alice        |2026-02-01 09:15:00|700   |0                |1                      |
|104     |C002       |Bob          |2026-01-07 11:00:00|200   |1                |0                      |
|105     |C002       |Bob          |2026-01-20 16:45:00|400   |0                |1                      |
|106     |C003       |Charlie      |2026-02-05 13:00:00|900   |1                |1                      |
+--------+-----------+-------------+----------

In [101]:
purchases_df.createOrReplaceTempView("purchases")

In [102]:
sql7="""
WITH ranked AS (
    SELECT
        order_id,
        customer_id,
        customer_name,
        purchase_time,
        amount,

        ROW_NUMBER() OVER (
            PARTITION BY customer_id
            ORDER BY purchase_time ASC, order_id ASC
        ) AS first_purchase_rn,

        ROW_NUMBER() OVER (
            PARTITION BY customer_id
            ORDER BY purchase_time DESC, order_id DESC
        ) AS latest_purchase_rn

    FROM purchases
)

SELECT
    order_id,
    customer_id,
    customer_name,
    purchase_time,
    amount,

    CASE
        WHEN first_purchase_rn = 1 THEN 1
        ELSE 0
    END AS is_first_purchase,

    CASE
        WHEN latest_purchase_rn = 1 THEN 1
        ELSE 0
    END AS is_most_recent_purchase

FROM ranked
ORDER BY customer_id, purchase_time;
"""
spark.sql(sql7).show()

+--------+-----------+-------------+-------------------+------+-----------------+-----------------------+
|order_id|customer_id|customer_name|      purchase_time|amount|is_first_purchase|is_most_recent_purchase|
+--------+-----------+-------------+-------------------+------+-----------------+-----------------------+
|     101|       C001|        Alice|2026-01-05 10:00:00|   500|                1|                      0|
|     102|       C001|        Alice|2026-01-10 14:30:00|   300|                0|                      0|
|     103|       C001|        Alice|2026-02-01 09:15:00|   700|                0|                      1|
|     104|       C002|          Bob|2026-01-07 11:00:00|   200|                1|                      0|
|     105|       C002|          Bob|2026-01-20 16:45:00|   400|                0|                      1|
|     106|       C003|      Charlie|2026-02-05 13:00:00|   900|                1|                      1|
+--------+-----------+-------------+----------